# 10b — Differential Accessibility

**Goal:** Run DESeq2 per cell type using two complementary strategies:
1. **Shared peaks (one-vs-all)** — peaks present in all species, each species vs all others
2. **Evolutionary branches (orthology-aware)** — phylogenetic, ILS, pairwise contrasts using
   only peaks with orthologous sequence in the compared species (not just all-species shared)

Plus **ultra-robust filtering** (`min(pos donors) > max(neg donors)` in logCPM space).

**Requires:** Intermediate files from notebook `10a`.

**Outputs:**
- `differential_results/shared_peaks/` — per-cell-type CSV + `DESeq2_res_list_shared.rds`
- `differential_results/evolutionary_branches/` — per cell-type × contrast CSVs + `branch_res_list.rds`
- `differential_results/ultra_robust_branches/` — ultra-robust filtered peaks per contrast
- `plots/{CellType}_Visualizations/` — volcano & heatmap PDFs
- `plots/Cross_CellType_*_Heatmap.pdf` — summary heatmaps
- `plots/Evolutionary_Staircase_*.pdf` — branch staircase heatmaps
- `differential_results/bed_files/` — BED files for significant peaks

In [1]:
# =============================================================================
# Cell 1: Load Packages & Source Utilities
# =============================================================================
suppressPackageStartupMessages({
  library(DESeq2)
  library(ggplot2)
  library(ggrepel)
  library(pheatmap)
  library(viridis)
  library(ComplexHeatmap)
  library(circlize)
  library(matrixStats)
  library(dplyr)
  library(tibble)
  library(readr)
})

source("../src/deseq2_utils.R")
message("Packages loaded & utilities sourced.")

Packages loaded & utilities sourced.



In [2]:
# =============================================================================
# Cell 2: Configuration & Load Intermediate Data
# =============================================================================
BASE      <- "/links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks"
OUT_DIR   <- file.path(BASE, "cross_species_consensus_v3/13_deseq2_R_pseudobulk")
SPECIES   <- c("Human", "Bonobo", "Chimpanzee", "Gorilla", "Macaque", "Marmoset")

save_dir <- file.path(OUT_DIR, "pseudobulk")

# Load shared-peak data (for Strategy 1)
pb_shared      <- readRDS(file.path(save_dir, "pb_data_shared.rds"))
counts_shared  <- pb_shared$counts
meta_shared    <- pb_shared$meta

# Load union-peak data (for Strategy 2: evolutionary branches)
pb_union       <- readRDS(file.path(save_dir, "pb_data_union.rds"))
counts_union   <- pb_union$counts

# Load global VST (for heatmaps)
vsd_global     <- readRDS(file.path(save_dir, "vsd_global_shared.rds"))

# Load master annotation & build orthology index
anno_file    <- file.path(BASE, "cross_species_consensus_v3/07_master_annotation/master_annotation_final.tsv")
master_anno  <- read_tsv(anno_file, show_col_types = FALSE)
options(scipen = 999)

ortho_mat <- build_orthology_index(master_anno, SPECIES)

message("Shared peaks: ", nrow(counts_shared), " x ", ncol(counts_shared))
message("Union  peaks: ", nrow(counts_union), " x ", ncol(counts_union))

Orthology index: 1142003 peaks x 6 species



Shared peaks: 71663 x 241



Union  peaks: 1142003 x 241



## Strategy 1: Shared Peaks (Conservative)

Test differential accessibility using only peaks present in **all** species.

In [3]:
# =============================================================================
# Cell 3: DESeq2 — Shared Peaks (All-vs-One Contrasts)
# =============================================================================
res_list_shared <- run_deseq2_shared_peaks(
  counts_shared, meta_shared, SPECIES, OUT_DIR
)

message("\nShared-peak DESeq2 complete for ", length(res_list_shared), " cell types.")


=== [Shared Peaks] Colonocytes ===



converting counts to integer mode



  Marmoset     specific: 4575 (padj<0.05, LFC>1)



  Human        specific: 4856 (padj<0.05, LFC>1)



  Gorilla      specific: 716 (padj<0.05, LFC>1)



...

  Chimpanzee   specific: 2454 (padj<0.05, LFC>1)



  Bonobo       specific: 3577 (padj<0.05, LFC>1)



  Macaque      specific: 4779 (padj<0.05, LFC>1)




Checkpoint: shared-peaks res_list saved.




Shared-peak DESeq2 complete for 13 cell types.



## Strategy 2: Evolutionary Branch Contrasts (Orthology-Aware)

For each contrast (phylogenetic node, ILS topology, pairwise), use only peaks
where **all involved species have physical orthologous sequence**. E.g. for
Human vs Gorilla, we use all peaks with orthology in both Human AND Gorilla
(not just peaks shared across all 6 species).

Then apply **ultra-robust filtering**: `min(positive donors in logCPM) > max(negative donors in logCPM)`.

In [4]:
# =============================================================================
# Cell 4: Define Evolutionary Contrasts & Run DESeq2
# =============================================================================
evo_contrasts <- define_evolutionary_contrasts()

branch_res <- run_deseq2_evolutionary(
  counts_union, meta_shared, evo_contrasts, ortho_mat, OUT_DIR, min_samples = 2
)

message("\nEvolutionary branch DESeq2 complete for ",
        length(branch_res), " cell types.")

Defined 19 evolutionary contrasts.




=== [Evolutionary Branches] Colonocytes ===



converting counts to integer mode



  Node1_Human_vs_Pan                      : 221769 peaks tested, 3620 sig UP



converting counts to integer mode



...

converting counts to integer mode



  Node_Apes_vs_Monkeys                    : 779816 peaks tested, 52759 sig UP




Outlier filtering: no samples removed across all contrasts.




Checkpoint: evolutionary branch results saved.




Evolutionary branch DESeq2 complete for 13 cell types.



In [5]:
# =============================================================================
# Cell 5: Ultra-Robust Filtering (min(pos) > max(neg) in logCPM)
# =============================================================================
robust_peaks <- ultra_robust_filter(
  branch_res, counts_union, meta_shared, evo_contrasts, OUT_DIR,
  padj_thresh = 0.05, lfc_thresh = 1, min_logcpm = 1
)

# Quick summary
n_total <- sum(sapply(robust_peaks, function(ct) sum(sapply(ct, length))))
message("\nTotal ultra-robust peaks across all cell types × contrasts: ", n_total)


Ultra-robust filtering: Colonocytes



  [Node1_Human_vs_Pan] DESeq2 UP: 3620 <e2><86><92> Ultra-Robust: 2754



  [Node2_AfricanApes_vs_Gorilla] DESeq2 UP: 253 <e2><86><92> Ultra-Robust: 253



  [Node3_GreatApes_vs_Macaque] DESeq2 UP: 3561 <e2><86><92> Ultra-Robust: 2922



  [Node4_OldWorld_vs_Marmoset] DESeq2 UP: 5996 <e2><86><92> Ultra-Robust: 2989



...

  [Pair_Human_vs_Marmoset] DESeq2 UP: 116349 <e2><86><92> Ultra-Robust: 36643



  [Div_Human_vs_AllPrimates] DESeq2 UP: 83637 <e2><86><92> Ultra-Robust: 2135



  [Node_Apes_vs_Monkeys] DESeq2 UP: 52759 <e2><86><92> Ultra-Robust: 1912




Ultra-robust filtering complete. Saved checkpoint.




Total ultra-robust peaks across all cell types <U+00D7> contrasts: 855083



## Volcano Plots

Volcano plots for each cell type × species (shared-peak strategy), plus
selected evolutionary branch contrasts. Top peak IDs labeled on both sides.

In [6]:
# =============================================================================
# Cell 6: Volcano Plots — Shared Peaks + Selected Branches
# =============================================================================
# A) Shared-peak volcanos (one per cell type × species)
for (ct in names(res_list_shared)) {
  for (sp in names(res_list_shared[[ct]])) {
    res_df <- as.data.frame(res_list_shared[[ct]][[sp]])
    volcano_dir <- file.path(OUT_DIR, "plots", paste0(ct, "_Visualizations"))
    out_file    <- file.path(volcano_dir,
                             paste0("Volcano_", sp, "_shared_", ct, ".pdf"))
    plot_volcano(res_df,
                 title    = paste(sp, "vs All (", ct, ") [shared peaks]"),
                 out_file = out_file, n_label = 10)
  }
}

# B) Branch volcanos (one per cell type × contrast)
for (ct in names(branch_res)) {
  for (node in names(branch_res[[ct]])) {
    res_df <- as.data.frame(branch_res[[ct]][[node]])
    volcano_dir <- file.path(OUT_DIR, "plots", paste0(ct, "_Visualizations"))
    out_file    <- file.path(volcano_dir,
                             paste0("Volcano_", node, "_branch_", ct, ".pdf"))
    plot_volcano(res_df,
                 title    = paste(node, "(", ct, ") [evolutionary branch]"),
                 out_file = out_file, n_label = 10)
  }
}

message("\nAll volcano plots generated (shared + branch).")

  Volcano saved: Volcano_Marmoset_shared_Colonocytes.pdf



  Volcano saved: Volcano_Human_shared_Colonocytes.pdf



  Volcano saved: Volcano_Gorilla_shared_Colonocytes.pdf



  Volcano saved: Volcano_Chimpanzee_shared_Colonocytes.pdf



  Volcano saved: Volcano_Bonobo_shared_Colonocytes.pdf



...

  Volcano saved: Volcano_Pair_Human_vs_Macaque_branch_TA.cells.pdf



  Volcano saved: Volcano_Pair_Human_vs_Marmoset_branch_TA.cells.pdf



  Volcano saved: Volcano_Div_Human_vs_AllPrimates_branch_TA.cells.pdf



  Volcano saved: Volcano_Node_Apes_vs_Monkeys_branch_TA.cells.pdf




All volcano plots generated (shared + branch).



## Species Marker Heatmaps

In [7]:
# =============================================================================
# Cell 6: Species Marker Heatmaps (Using Shared-Peak Results)
# =============================================================================
for (ct in names(res_list_shared)) {
  ct_plot_dir <- file.path(OUT_DIR, "plots", paste0(ct, "_Visualizations"))
  heatmap_file <- file.path(ct_plot_dir, paste0("Heatmap_Top_Markers_", ct, ".pdf"))

  plot_species_heatmap(res_list_shared[[ct]], vsd_global, meta_shared,
                       cell_type = ct, out_file = heatmap_file, top_n = 25)
}

message("All species marker heatmaps generated.")

  Marmoset: 4575 sig peaks, top 25 selected



  Human: 4856 sig peaks, top 25 selected



  Gorilla: 716 sig peaks, top 25 selected



  Chimpanzee: 159 sig peaks, top 25 selected



  Bonobo: 130 sig peaks, top 25 selected



...

  Macaque: 4779 sig peaks, top 25 selected



  Total unique top regions: 145



  After VST intersect: 145 regions



  Heatmap saved: Heatmap_Top_Markers_TA.cells.pdf



All species marker heatmaps generated.



## Cross-Cell-Type Summary Heatmaps

In [8]:
# =============================================================================
# Cell 7: Cross-Cell-Type Signed P-Value Heatmap (Shared Peaks)
# =============================================================================
plots_dir <- file.path(OUT_DIR, "plots")

# Shared-peak heatmap (Human vs rest, using peaks shared across all species)
plot_mat_shared <- plot_cross_celltype_heatmap(
  res_list_shared, target_sp = "Human",
  out_file = file.path(plots_dir, "Cross_CellType_Human_Heatmap_shared.pdf")
)

# Save signed-pvalue matrix for module extraction in notebook 10c
saveRDS(plot_mat_shared,
        file.path(OUT_DIR, "differential_results", "Signed_Pval_Matrix_shared.rds"))

message("Cross-cell-type heatmap and signed-p matrix saved.")

Regions significant in >= 1 cell type: 27131



`use_raster` is automatically set to TRUE for a matrix with more than
2000 rows. You can control `use_raster` argument by explicitly setting
TRUE/FALSE to it.

Set `ht_opt$message = FALSE` to turn off this message.



Cross-cell-type heatmap saved: Cross_CellType_Human_Heatmap_shared.pdf



Cross-cell-type heatmap and signed-p matrix saved.



## Evolutionary Staircase Heatmap

Visualize the ultra-robust peaks from evolutionary branch testing: for each node,
show the top peaks with Z-scored average expression per species. Produces one
heatmap per cell type.

In [9]:
# =============================================================================
# Cell 9: Evolutionary Staircase Heatmap (Ultra-Robust Peaks)
# =============================================================================
species_order <- c("Human", "Chimpanzee", "Bonobo", "Gorilla", "Macaque", "Marmoset")

# Node order for display (evolutionary steps)
node_order <- c("Node1_Human_vs_Pan", "Node2_AfricanApes_vs_Gorilla",
                "Node3_GreatApes_vs_Macaque", "Node4_OldWorld_vs_Marmoset",
                "Div_Human_vs_Apes")

for (ct in names(robust_peaks)) {
  ct_nodes <- intersect(node_order, names(robust_peaks[[ct]]))
  ct_nodes <- ct_nodes[sapply(ct_nodes, function(n) length(robust_peaks[[ct]][[n]]) > 0)]
  if (length(ct_nodes) == 0) next

  # Get counts for this cell type
  meta_ct    <- meta_shared[meta_shared$cell_type == ct, ]
  ct_samples <- intersect(rownames(meta_ct), colnames(counts_union))
  ct_counts  <- counts_union[, ct_samples, drop = FALSE]
  meta_ct    <- meta_ct[ct_samples, ]

  lib_sizes  <- colSums(ct_counts)
  cpm_mat    <- t(t(ct_counts) / lib_sizes) * 1e6
  logcpm_mat <- log2(cpm_mat + 1)

  # Average expression per species
  avg_exp <- matrix(NA, nrow = nrow(logcpm_mat), ncol = length(species_order),
                    dimnames = list(rownames(logcpm_mat), species_order))
  for (sp in species_order) {
    sp_samples <- ct_samples[meta_ct$species == sp]
    if (length(sp_samples) > 1)
      avg_exp[, sp] <- rowMeans(logcpm_mat[, sp_samples, drop = FALSE])
    else if (length(sp_samples) == 1)
      avg_exp[, sp] <- logcpm_mat[, sp_samples]
  }

  # Collect top peaks per branch
  plot_peaks <- c()
  row_splits <- c()

  for (node in ct_nodes) {
    ur_peaks <- robust_peaks[[ct]][[node]]
    if (length(ur_peaks) == 0) next

    # Sort by LFC from DESeq2 result, take top 50
    res_df <- as.data.frame(branch_res[[ct]][[node]])
    ur_res <- res_df[intersect(ur_peaks, rownames(res_df)), , drop = FALSE]
    sorted <- rownames(ur_res)[order(ur_res$log2FoldChange, decreasing = TRUE)]
    top_peaks <- head(sorted, 50)

    plot_peaks <- c(plot_peaks, top_peaks)
    row_splits <- c(row_splits, rep(node, length(top_peaks)))
  }

  if (length(plot_peaks) < 5) next

  # Z-score matrix
  mat <- avg_exp[plot_peaks, , drop = FALSE]
  valid_cols <- apply(mat, 2, function(x) !all(is.na(x)))
  mat <- mat[, valid_cols, drop = FALSE]
  mat_scaled <- t(apply(mat, 1, scale))
  colnames(mat_scaled) <- colnames(mat)
  split_factor <- factor(row_splits, levels = ct_nodes)

  col_fun <- colorRamp2(c(-2, 0, 2), c("#377eb8", "white", "#e41a1c"))

  ht <- Heatmap(mat_scaled, name = "Z-score", col = col_fun,
                row_split = split_factor, row_title_rot = 0,
                row_title_gp = gpar(fontsize = 9, fontface = "bold"),
                row_gap = unit(2, "mm"), cluster_rows = TRUE,
                cluster_columns = FALSE,
                show_row_names = FALSE, show_column_names = TRUE,
                column_names_rot = 45,
                column_title = paste("Ultra-Robust Evolutionary Staircase:", ct),
                column_title_gp = gpar(fontsize = 14, fontface = "bold"),
                heatmap_legend_param = list(title = "Z-score"))

  ht_file <- file.path(OUT_DIR, "plots",
                        paste0("Evolutionary_Staircase_", ct, ".pdf"))
  dir.create(dirname(ht_file), showWarnings = FALSE, recursive = TRUE)
  dynamic_h <- max(8, length(plot_peaks) * 0.08 + length(ct_nodes))
  pdf(ht_file, width = 10, height = dynamic_h)
  draw(ht, merge_legend = TRUE)
  dev.off()
  message("  Staircase heatmap: ", ct, " (", length(plot_peaks), " peaks)")
}

message("\nEvolutionary staircase heatmaps complete.")

  Staircase heatmap: Colonocytes (250 peaks)



  Staircase heatmap: Crypt.Fibroblasts.WNT2B. (250 peaks)



  Staircase heatmap: ECs (151 peaks)



  Staircase heatmap: Enterocytes (250 peaks)



  Staircase heatmap: Goblet.cells (250 peaks)



...

  Staircase heatmap: Specialized.Fibroblasts.SYNM. (250 peaks)



  Staircase heatmap: Stem.cells (250 peaks)



  Staircase heatmap: T.cells (250 peaks)



  Staircase heatmap: TA.cells (250 peaks)




Evolutionary staircase heatmaps complete.



## BED File Generation

In [10]:
# =============================================================================
# Cell 10: Generate BED Files (Shared + Branch + Ultra-Robust)
# =============================================================================

# --- A) Shared-peak BED files (Human UP / DOWN) ---
bed_base_shared <- file.path(OUT_DIR, "differential_results", "shared_peaks")
all_human_up_shared <- c()

for (ct in names(res_list_shared)) {
  if (!"Human" %in% names(res_list_shared[[ct]])) next
  res_df <- as.data.frame(res_list_shared[[ct]][["Human"]])

  up_peaks   <- rownames(res_df)[!is.na(res_df$padj) &
                                  res_df$padj < 0.05 &
                                  res_df$log2FoldChange > 1]
  down_peaks <- rownames(res_df)[!is.na(res_df$padj) &
                                  res_df$padj < 0.05 &
                                  res_df$log2FoldChange < -1]

  all_human_up_shared <- c(all_human_up_shared, up_peaks)

  ct_bed_dir <- file.path(bed_base_shared, ct)
  if (length(up_peaks) > 0)
    write_peaks_bed(up_peaks, "Human", master_anno,
                    file.path(ct_bed_dir, "Human_UP_Regions.bed"))
  if (length(down_peaks) > 0)
    write_peaks_bed(down_peaks, "Human", master_anno,
                    file.path(ct_bed_dir, "Human_DOWN_Regions.bed"))
}

# Global BED of all Human-UP regions (shared)
bed_dir <- file.path(OUT_DIR, "differential_results", "bed_files")
dir.create(bed_dir, showWarnings = FALSE, recursive = TRUE)
write_peaks_bed(unique(all_human_up_shared), "Human", master_anno,
                file.path(bed_dir, "All_Human_UP_shared.bed"))

# --- B) Evolutionary branch BED files (significant per node per cell type) ---
bed_base_branch <- file.path(OUT_DIR, "differential_results", "evolutionary_branches")
for (ct in names(branch_res)) {
  for (node in names(branch_res[[ct]])) {
    res_df <- as.data.frame(branch_res[[ct]][[node]])
    sig_peaks <- rownames(res_df)[!is.na(res_df$padj) & res_df$padj < 0.05]
    if (length(sig_peaks) == 0) next

    ct_bed_dir <- file.path(bed_base_branch, ct)
    write_peaks_bed(sig_peaks, "Human", master_anno,
                    file.path(ct_bed_dir, paste0(node, "_sig.bed")))
  }
}

# --- C) Ultra-robust BED files ---
bed_base_robust <- file.path(OUT_DIR, "differential_results", "ultra_robust_branches")
for (ct in names(robust_peaks)) {
  for (node in names(robust_peaks[[ct]])) {
    ur_peaks <- robust_peaks[[ct]][[node]]
    if (length(ur_peaks) == 0) next

    ct_bed_dir <- file.path(bed_base_robust, ct)
    write_peaks_bed(ur_peaks, "Human", master_anno,
                    file.path(ct_bed_dir, paste0(node, "_ultra_robust.bed")))
  }
}

message("\nAll BED files generated (shared + branch + ultra-robust).")

  Saved 4856 regions to: Human_UP_Regions.bed



  Saved 1721 regions to: Human_DOWN_Regions.bed



  Saved 2638 regions to: Human_UP_Regions.bed



  Saved 1518 regions to: Human_DOWN_Regions.bed



  Saved 171 regions to: Human_UP_Regions.bed



...

  Saved 28210 regions to: Pair_Human_vs_Macaque_ultra_robust.bed



  Saved 36643 regions to: Pair_Human_vs_Marmoset_ultra_robust.bed



  Saved 2135 regions to: Div_Human_vs_AllPrimates_ultra_robust.bed



  Saved 1912 regions to: Node_Apes_vs_Monkeys_ultra_robust.bed




All BED files generated (shared + branch + ultra-robust).



In [11]:
# =============================================================================
# Cell 11: Save Final Checkpoint
# =============================================================================
diff_dir <- file.path(OUT_DIR, "differential_results")

saveRDS(res_list_shared,
        file.path(diff_dir, "DESeq2_res_list_shared_all.rds"))
# branch_res_list.rds and ultra_robust_peaks_list.rds already saved by functions

message("\n=== Notebook 10b complete ===")
message("Shared-peak results:       ", length(res_list_shared), " cell types")
message("Branch results:            ", length(branch_res), " cell types")
message("Ultra-robust peak sets:    ", length(robust_peaks), " cell types")
message("\nSaved to: ", diff_dir)


=== Notebook 10b complete ===



Shared-peak results:       13 cell types



Branch results:            13 cell types



Ultra-robust peak sets:    13 cell types




Saved to: /links/groups/treutlein/USERS/jjans/analysis/adult_intestine/peaks/cross_species_consensus_v3/13_deseq2_R_pseudobulk/differential_results

